In [1]:
#!/usr/bin/env python

# Program: Mouse sequence assembly
# Authors: Spencer Todd, Thu Thu Han, & Stefanie Moreno
# Date: February 25, 2026

# Import packages
import csv
import gzip
import random
import time

# Import modules
from collections import defaultdict, Counter
from datetime import datetime


def read_fastq_and_build_graph(filename, dbg, k):
    """
    Unzip fastq file containing 10 million "perfect" simulated 150bp reads from M. musculus.
    Stream reads line by line to build subgraph for each sequence with DeBruijnGraph object's
    `build_graph_from_reads` method. This approach avoids storing 10M reads in list format
     Parameters:
         filename (str): filename of gzipped fastq file
         dbg (instance of DeBruijnGraph class): graph object being actively built as streaming in reads
         k (int): kmer length
    Returns:
        read_count (int)
        total_bases (int)
        first_read (str)
        last_read (str)
    """

    # Initialize counts
    read_count = 0
    total_bases = 0
    first_read = None
    last_read = None
    line_count = 0

    with gzip.open(filename, 'rt') as f:
        for line in f:
            line_count += 1

            # Remove newline and store DNA sequences from line 2 of fastq record
            if line_count % 4 == 2:
                read = line.strip()

                if first_read is None:
                    first_read = read  # Save first read encountered
                last_read = read  # Always update so ends up being final read

                read_count += 1  # Increment read count
                total_bases += len(read)  # Add read to all seen bases running total

                dbg.build_graph_from_reads([read], k)  # Add all kmers from read into DeBruijn graph

    return read_count, total_bases, first_read, last_read


class DeBruijnGraph:
    """
    Main class for De Bruijn graphs and genome assembly
    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of graph.
    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges
         Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers
        k (int): The k-mer size used for graph construction
    """


    def __init__(self, reads, k):
        """
        Initialize De Bruijn graph from sequencing reads
        Arguments:
            reads (list): List of DNA sequence strings
            k (int): K-mer size for graph construction
        """
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)


    def add_edge(self, left, right):
        """
        Add directed edge to the graph.
        Args:
            left (str): Source (k-1)-mer node
            right (str): Destination (k-1)-mer node
            """
        self.graph[left].append(right)
        if right not in self.graph:
            self.graph[right] = []


    def remove_edge(self, left, right):
        """
        Remove a directed edge from the graph.
        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node
            """
        if left in self.graph:
            try:
                self.graph[left].remove(right)
                if not self.graph[left]:
                    del self.graph[left]
            except ValueError:
                pass


    def build_graph_from_reads(self, reads, k):
        """
        Build De Bruijn graph from multiple sequencing reads.
        This method supports building graphs in batch mode even
        though our pipeline streams one read at a time. This is
        still essential though since open fastq calls this with
        a single read list.
        """
        if k < 2:
            raise ValueError("k must be >= 2")

        # Process each read in list
        for read in reads:
            # Skip reads shorter too short to contain one full kmer
            if len(read) < k:
                continue

            # Slide window of length k across read to extract all kmerd
            for i in range(len(read) - k + 1):
                kmer = read[i:i + k]
                # Split k‑mer into (k‑1)-mer prefix and suffix to form a directed edge
                prefix = kmer[:-1]
                suffix = kmer[1:]
                # Add edge prefix → suffix to the De Bruijn graph
                self.add_edge(prefix, suffix)


    def eulerian_walk(self, node, graph, seed=None):
        """Perform recursive Eulerian walk on graph component
        This is recursive function that follow all edges from a node
        to traverse the graph, building path in reverse order
        Arguments:
            node (str): Current node to traverse from
            graph (defaultdict): Graph or subgraph to traverse
            seed (int, optional): Seed for random edge selection
        Returns:
            list: List of (k-1)-mers traversed (in reverse order)
            """
        # Set rng seed so edge choices are reproducible
        if seed is not None:
            random.seed(seed)

        # Initialize lists for Eulerian path and stack for recursive traversal
        path = []  # List to store final Eulerian path
        stack = [node]  # Stack simulating recursive traversal

        while stack:
            # Look at current node on top of stack
            v = stack[-1]

            if graph[v]:  # If node still has outgoing edges
                if seed is not None and len(graph[v]) > 1:
                    # Use randomizer to choose next node if multiple options
                    w = random.choice(graph[v])
                else:
                    # Otherwise take last outgoing edge
                    w = graph[v][-1]

                # Remove edge v -> w
                graph[v].remove(w)
                # Move to next node
                stack.append(w)

            else:
                # No outgoing edges: add node to path and backtrack
                path.append(stack.pop())

        # Reverse since nodes added in reverse order
        path.reverse()
        # Return Eulerian path as list of kmer
        return path


    def assemble_contigs(self, seed=None):
        """
        Assemble all contigs from the De Bruijn graph
        Finds all connected components and generates Eulerian path
        for each component, producing multiple assembled contigs.
        Arguments:
            seed (int): Random seed for reproducible assembly
        Returns:
            list (str): List of assembled DNA contig sequences
        """
        indeg = Counter()  # Track edges pointing into each node
        outdeg = Counter()  # Track edges pointing out of each node


        # Compute in-degree and out-degree for each node in graph
        for u, nbrs in self.graph.items():
            outdeg[u] += len(nbrs)  # number of outgoing edges from u
            for v in nbrs:
                indeg[v] += 1  # Each neighbor v gets one incoming edge

        nodes = set(indeg.keys()) | set(outdeg.keys())  # All nodes appearing in graph
        unused = set(nodes)  # Nodes not yet assigned to a component
        contigs = []  # Store assembled contigs

        while unused:
            comp_nodes = set()  # Nodes in current connected component
            stack = [next(iter(unused))]  # Start DFS from arbitrary unused

            # DFS to collect all nodes in connected component
            while stack:
                u = stack.pop()
                if u in comp_nodes:
                    continue
                comp_nodes.add(u)

                # Follow forward edges
                for v in self.graph.get(u, []):
                    if v not in comp_nodes:
                        stack.append(v)

                # Follow reverse edges to ensure full connectivity
                for x, nbrs in self.graph.items():
                    if u in nbrs and x not in comp_nodes:
                        stack.append(x)

            # Choose good Eulerian start node
            # Prefer nodes where outdeg = indeg + 1 (Eulerian path start)
            candidates = [u for u in comp_nodes if outdeg[u] == indeg[u] + 1]
            if candidates:
                start = candidates[0]

            else:
                # Otherwise pick any node with outgoing edges, or any node
                with_out = [u for u in comp_nodes if outdeg[u] > 0]
                start = with_out[0] if with_out else next(iter(comp_nodes))

            # Build subgraph containing only nodes in this component
            subgraph = defaultdict(list)
            for u in comp_nodes:
                subgraph[u] = list(self.graph.get(u, []))

            # Perform Eulerian walk on this component
            tour = self.eulerian_walk(start, subgraph, seed=seed)

            # Convert Eulerian tour into contig sequence
            if len(tour) > 1:
                contig = self.tour_to_sequence(tour)
                contigs.append(contig)

            unused -= comp_nodes

        return contigs


    def tour_to_sequence(self, tour):
        """
        Convert a tour of (k-1)-mers into a DNA sequence
        Argument:
            tour (list): List of (k-1)-mer strings in order
        Returns:
            str: Assembled DNA sequence
        """
        if not tour:
            return ""
        seq = [tour[0]]
        for node in tour[1:]:
            seq.append(node[-1])
        return "".join(seq)

    def get_assembly_stats(self, contigs):
        """
        Calculate assembly statistics for assembled contigs
        Argument:
            contigs (list): List of contig sequences
        Returns:
            dict: Dictionary containing assembly statistics:
                num_contigs: Total number of contigs
                total_length: Total assembled sequence length
                longest_contig: Length of longest contig
                shortest_contig: Length of shortest contig
                mean_length: Mean contig length
                -n50: N50 statistic
        """
        if not contigs:
            return {
                "num_contigs": 0,
                "total_length": 0,
                "longest_contig": 0,
                "shortest_contig": 0,
                "mean_length": 0,
                "n50": 0,
            }

        lengths = sorted((len(c) for c in contigs), reverse=True)
        total = sum(lengths)
        num = len(lengths)
        longest = lengths[0]
        shortest = lengths[-1]
        mean_len = total / num

        running = 0
        half = total / 2
        n50 = 0
        for L in lengths:
            running += L
            if running >= half:
                n50 = L
                break

        return {
            "num_contigs": num,
            "total_length": total,
            "longest_contig": longest,
            "shortest_contig": shortest,
            "mean_length": mean_len,
            "n50": n50,
        }


    def write_fasta(self, contigs, filename):
        """
        Write assembled contigs to FASTA file
        """
        with open(filename, "w") as fh:
            for i, seq in enumerate(contigs, start=1):
                fh.write(f">contig_{i}\n")
                for j in range(0, len(seq), 80):
                    fh.write(seq[j:j+80] + "\n")


    def write_contig_csv(self, contigs, csv_filename, coverage_estimate):
        """
        Write contig metadata (ID, start position, coverage) to CSV
        """
        with open(csv_filename, "w", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["contig_id", "start_position", "coverage"])

            for i, seq in enumerate(contigs, start=1):
                writer.writerow([f"contig_{i}", 1, f"{coverage_estimate:.2f}"])


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    random_seed,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    coverage_estimate,
    assembly_fraction
):
    """
    Write comprehensive assembly statistics to text file
    """
    with open(stats_file, "w") as f:
        f.write("MOUSE GENOME ASSEMBLY STATISTICS\n\n")
        f.write(f"Input File: {input_file}\n")
        f.write(f"K-mer Size: {k_mer_size}\n")
        f.write(f"Random Seed: {random_seed}\n\n")

        f.write("INPUT DATA\n")
        f.write(f"Number of reads: {num_reads:,}\n")
        f.write(f"Average read length: {avg_read_length:.1f} bp\n")
        f.write(f"Estimated coverage: {coverage_estimate:.1f}x\n\n")

        f.write("GRAPH STATISTICS\n")
        f.write(f"Nodes: {num_nodes:,}\n")
        f.write(f"Edges: {num_edges:,}\n\n")

        f.write("ASSEMBLY RESULTS\n")
        f.write(f"Number of contigs: {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length: {stats['total_length']:,} bp\n")
        f.write(f"N50: {stats['n50']:,} bp\n")
        f.write(f"Assembly fraction: {assembly_fraction:.2f}%\n\n")

        f.write("TOP 20 CONTIG LENGTHS\n")
        for i, L in enumerate(contig_lengths[:20], 1):
            f.write(f"{i}. {L:,} bp\n")


# Driver program to execute script
if __name__ == "__main__":

    input_file = "/Users/biotechiestefnie/Desktop/project04/mouse_SE_150bp.fq.gz"

    k_mer_size = 51
    random_seed = 42

    fasta_output = "mouse_contigs.fasta"
    csv_output = "mouse_contigs.csv"
    stats_output = "assembly_stats.txt"

    timing = {}

    dbg = DeBruijnGraph([], k_mer_size)

    t0 = time.time()
    num_reads, total_bases, first_read, last_read = read_fastq_and_build_graph(
        input_file, dbg, k_mer_size
    )
    timing["read_time"] = time.time() - t0

    avg_read_length = total_bases / num_reads if num_reads > 0 else 0
    coverage_estimate = total_bases / 2_600_000_000

    t1 = time.time()
    contigs = dbg.assemble_contigs(seed=random_seed)
    timing["assembly_time"] = time.time() - t1

    stats = dbg.get_assembly_stats(contigs)
    contig_lengths = sorted((len(c) for c in contigs), reverse=True)

    num_nodes = len(dbg.graph)
    num_edges = sum(len(v) for v in dbg.graph.values())

    assembly_fraction = (stats["total_length"] / 2_600_000_000) * 100

    dbg.write_fasta(contigs, fasta_output)
    dbg.write_contig_csv(contigs, csv_output, coverage_estimate)

    write_statistics_file(
        stats_output,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        random_seed,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        coverage_estimate,
        assembly_fraction
    )


FileNotFoundError: [Errno 2] No such file or directory: 'mouse_reads.fastq.gz'